# PROJECT 4
by Szymon Waliczek
 - Impact of spatial quantization and transverse confinement on 2DEG electron transport in **InAs**
 - Normal wire with leads
 - **With spin**
 - With Peierls phase - **QHE**

In [ ]:
import ipyparallel as ipp
cluster = ipp.Client(profile  = "kwant_parallel")
v = cluster[:]
lview = cluster.load_balanced_view()
len(v)

In [ ]:
%%px --local

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import kwant
import numpy as np
from scipy.sparse.linalg import eigs

In [ ]:
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

In [ ]:
%%px --local

# Pauli Matrices
import tinyarray
s_0 = tinyarray.array([[1, 0], [0, 1]])
s_x = tinyarray.array([[0, 1], [1, 0]])
s_y = tinyarray.array([[0, -1j], [1j, 0]])
s_z = tinyarray.array([[1, 0], [0, -1]])

In [ ]:
%%px --local

# Physical constants
from scipy.constants import physical_constants
eV = physical_constants['electron volt'][0]
m_el = physical_constants['electron mass'][0]
mu_B = physical_constants['Bohr magneton in eV/T'][0]
h_bar = physical_constants['Planck constant over 2 pi'][0]
phi_0 = physical_constants['elementary charge over h-bar'][0]
mu = 0
g_factor = -14.7
a = 5e-9
m_eff = 0.023 * m_el
t = (h_bar**2 / (2 * m_eff * a**2)) / eV # hopping in eV
W, L = 100, 200
freedom_deg = 2; # Two spins
PI = np.pi

In [ ]:
print(f"h_bar = {h_bar}")
print(f"m_eff = {m_eff}")
print(f"a = {a}")
print(f"t = {t}")
print(f"W, L = {W, L}")
print(f"phi_0 = {phi_0}")

In [ ]:
%%px --local

def onsite(site, mu, B):
    E_z = 0.5*g_factor*mu_B*B
    return (4*t - mu)*s_0 + E_z*s_z
#-----------------------------------------------------------------------
def hop(site1, site2, alpha, B):
    x1, y1 = site1.pos
    x2, y2 = site2.pos
    dx, dy = site1.pos - site2.pos
    x_step = (x1+x2) / 2 * 1e-9
    # Faza Peierlsa, Gauge: A = [0, Bx, 0])
    p_phase = np.exp(-1j *phi_0*B*x_step*(y1 - y2)*1e-9)
    
    if abs(dx) > a*0.5:
        return (-t * s_0 + (1j * alpha / (2 * a*1e9)) * s_y) * p_phase
    if abs(dy) > a*0.5:
        return (-t * s_0 - (1j * alpha / (2 * a*1e9)) * s_x) * p_phase

In [ ]:
%%px --local

def rectangle_fine(pos, width, length):
    (x, y) = pos
    return abs(x) < width/2 and abs(y) < length/2
#-----------------------------------------------------------------------------   
def make_sys_fine(a, width, length):
    lat = kwant.lattice.square(a*1e9, norbs = freedom_deg)
    sys = kwant.Builder()
    sys[lat.shape(lambda p: rectangle_fine(p, width, length), (0, 0))] = onsite
    sys[lat.neighbors(1)] = hop
    return sys, lat
#-----------------------------------------------------------------------------
def lead_shape(pos, width):
    return abs(pos[0]) < width / 2
#-----------------------------------------------------------------------------
def make_lead(sys, lat, a, width):
    lead_down = kwant.Builder(kwant.TranslationalSymmetry((0, -a*1e9)))
    lead_down[lat.shape(lambda p: lead_shape(p, width), (0, 0))] = onsite
    lead_down[lat.neighbors()] = hop
    lead_up = lead_down.reversed()
    sys.attach_lead(lead_down)
    sys.attach_lead(lead_up)

In [ ]:
def plot_sys(sys):
    kwant.plot(sys, fig_size=(2, 3.5), show=False)
    plt.title("fsys a = 5nm") 
    plt.xlabel("x [nm]")
    plt.ylabel("y [nm]")
    plt.show()

In [ ]:
%%px --local

sys_fine, lat_fine = make_sys_fine(a, W, L)
make_lead(sys_fine, lat_fine, a, W)
fsys = sys_fine.finalized()

In [ ]:
plot_sys(fsys)

In [ ]:
def plot_bands(lead, lead_name, ymin, ymax, xlim, params):        
    kwant.plotter.bands(lead, show=False, params=params, momenta=500, fig_size=(4,3))
    plt.ylabel("Energy [eV]")
    plt.xlabel("$k_y$ [$1/nm$]")
    plt.ylim(ymin,ymax)
    plt.xlim((-xlim) / (a*1e9), (xlim) / (a*1e9))
    plt.grid(True)
    plt.show()

In [ ]:
%%px --local

params = dict(mu=0, B=0, alpha=0)

In [ ]:
plot_bands(fsys.leads[0], 'lead IN', 0, 0.02, PI, params)

In [ ]:
%%px --local

def compute_transmission(energy, sys, params):
    smatrix = kwant.smatrix(sys, energy, params=params)
    return smatrix.transmission(1, 0)

In [ ]:
def plot_conductance(E, T, params):
    plt.figure(figsize=(3, 3))
    plt.plot(E, T, lw=1)
    plt.title(f"Conductance | B = {params['B']} T") 
    plt.xlabel("$E [eV]$")
    plt.ylabel("$G [2*e^2/h]$")
    plt.grid(True)
    plt.show()

In [ ]:
Evals = np.linspace(0, 0.05, 1000)

In [ ]:
Trans = lview.map_sync(lambda e: compute_transmission(e, fsys, params), Evals)

In [ ]:
plot_conductance(Evals, Trans, params)

In [ ]:
%%px --local

def compute_density_parallel(E, sys, params):
    modes = sys.leads[0].modes(energy=E, params=params)[0]
    channels = len(modes.momenta) // 2
    print(f"for energy = {E} | {channels} channels")
    res = lview.map_sync(lambda n: kwant.operator.Density(sys)(kwant.wave_function(sys, E, params=params)(0)[n]), range(channels))
    return sum(res)

In [ ]:
def plot_density_map(sys, E, params):
    psi2 = compute_density_parallel(E, sys, params=params)
    if psi2 is not None:
        kwant.plotter.map(sys, psi2, show=False, fig_size=(4, 3), vmin=0, vmax=max(psi2))
        plt.title(fr"$|\Psi|^2$ | $E = {E}$ eV")
        plt.xlabel("x [nm]"); plt.ylabel("y [nm]")
        plt.show()

In [ ]:
plot_density_map(fsys, E=0.02, params=params)